# ETL Project: E-commerce Data Pipeline


## Problem statement

You are a data engineer building a pipeline for an e-commerce system.

**Sources:**
- Users API (JSONPlaceholder)
- Orders (simulated API response — small fixed dataset)

**Goals:**
- Clean data
- Store in PostgreSQL
- Produce **analytics-ready** joins


## Pipeline architecture

```text
Users API  → transform → users table
Orders API → transform → orders table
              ↓
        SQL analytics (JOIN + aggregates)
```

👉 Separate **ingestion** from **consumption** — warehouse mindset.


## Prerequisites

- PostgreSQL (`../02_sql/README.md`).
- This notebook **`DROP`s** `orders` and `users`** if they exist** — backup anything you need from earlier lessons.
- Logs: `etl_project.log` (gitignored).


## DB setup


In [ ]:
import psycopg2
import requests

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin",
)

cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders CASCADE")
cur.execute("DROP TABLE IF EXISTS users CASCADE")

cur.execute(
    """
CREATE TABLE IF NOT EXISTS users (
    id INT PRIMARY KEY,
    name TEXT,
    email TEXT
)
"""
)

cur.execute(
    """
CREATE TABLE IF NOT EXISTS orders (
    id INT PRIMARY KEY,
    user_id INT,
    amount INT,
    created_at DATE
)
"""
)

conn.commit()
print("Tables users + orders ready.")


## Extract


In [ ]:
def extract_users():
    return requests.get("https://jsonplaceholder.typicode.com/users").json()


def extract_orders():
    # Simulated orders payload (replace with real API in production)
    return [
        {"id": 1, "user_id": 1, "amount": 200, "created_at": "2024-01-01"},
        {"id": 2, "user_id": 2, "amount": 500, "created_at": "2024-01-02"},
        {"id": 3, "user_id": 1, "amount": 300, "created_at": "2024-01-03"},
    ]


## Transform


In [ ]:
def transform_users(data):
    return [
        {
            "id": d["id"],
            "name": d["name"].title(),
            "email": d["email"].lower(),
        }
        for d in data
    ]


def transform_orders(data):
    return [
        {
            "id": d["id"],
            "user_id": d["user_id"],
            "amount": int(d["amount"]),
            "created_at": d["created_at"],
        }
        for d in data
    ]


## Load


In [ ]:
def load_users(data):
    for d in data:
        cur.execute(
            """
        INSERT INTO users (id, name, email)
        VALUES (%s, %s, %s)
        ON CONFLICT (id) DO NOTHING
        """,
            (d["id"], d["name"], d["email"]),
        )

    conn.commit()


def load_orders(data):
    for d in data:
        cur.execute(
            """
        INSERT INTO orders (id, user_id, amount, created_at)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (id) DO NOTHING
        """,
            (d["id"], d["user_id"], d["amount"], d["created_at"]),
        )

    conn.commit()


## Run pipeline


In [ ]:
def run_pipeline():
    users_raw = extract_users()
    orders_raw = extract_orders()

    users_clean = transform_users(users_raw)
    orders_clean = transform_orders(orders_raw)

    load_users(users_clean)
    load_orders(orders_clean)

    print("Pipeline executed successfully")


run_pipeline()


## Analytics query


In [ ]:
cur.execute(
    """
SELECT users.name,
SUM(orders.amount) as total_spent
FROM orders
JOIN users ON users.id = orders.user_id
GROUP BY users.name
ORDER BY total_spent DESC
"""
)

print(cur.fetchall())


## Logging + retry (production touches)

Reuse patterns from notebooks **03** and **04**: file logging, retries on HTTP extract, validate row counts before load.


In [ ]:
import logging
import time

logging.basicConfig(
    filename="etl_project.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)


def extract_users_with_retry(url=None, max_attempts=3, backoff_seconds=2):
    url = url or "https://jsonplaceholder.typicode.com/users"
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            logging.info("extract_users succeeded on attempt %s", attempt)
            return response.json()
        except Exception as exc:
            logging.warning("extract_users attempt %s failed: %s", attempt, exc)
            if attempt == max_attempts:
                logging.error("extract_users exhausted retries")
                return []
            time.sleep(backoff_seconds)
    return []


def run_pipeline_logged():
    logging.info("pipeline started")

    users_raw = extract_users_with_retry()
    orders_raw = extract_orders()

    if not users_raw:
        logging.error("no users extracted — aborting load")
        return

    users_clean = transform_users(users_raw)
    orders_clean = transform_orders(orders_raw)

    logging.info("loaded user rows=%s order rows=%s", len(users_clean), len(orders_clean))

    load_users(users_clean)
    load_orders(orders_clean)

    logging.info("pipeline completed")


# Idempotent: safe after run_pipeline(); uses ON CONFLICT
run_pipeline_logged()


## Assignment

Upgrade the pipeline:

- **Incremental** loading (high-water mark table — see `04_incremental_pipeline.ipynb`)
- **Logging** file + structured messages
- **Validation** (reject bad rows, assert FK-like user exists)
- **Error handling** around each stage

**Bonus:** `products` table + revenue per product.


## Interview Questions

1. How do you design an ETL pipeline?
2. How do you handle failures?
3. How do you scale pipelines?
4. How do you ensure data quality?


## What you just built

- **Multi-source** ingestion (API + simulated orders)
- **Transforms** + **relational load**
- **Analytics** SQL on curated tables

👉 Interview story: *“End-to-end ETL for e-commerce — ingest, quality, serve analytics.”*


---

### Phase 3 complete

You can chain extract → transform → load → incremental patterns with observability.

**Next — Phase 4:** **Big Data / Spark** — distributed execution and scale-out processing.


## Optional: close connection


In [ ]:
cur.close()
conn.close()
print("Connection closed.")


## Your tasks

- [ ] Run **DB setup** → **Run pipeline** → **Analytics query** (network on).
- [ ] Run **Logging + retry** cell; inspect `etl_project.log`.
- [ ] **Assignment:** incremental table + validation + error boundaries; **bonus:** products + revenue.
- [ ] Write your **portfolio README** paragraph (sources, guarantees, how you’d schedule this).
- [ ] Prepare **two** failure scenarios you’d test (API 500, duplicate PK, schema drift).
